# 07 — Topic Modeling

BERTopic on cached sentence embeddings, with an LDA pass for comparison. KeyBERT extracts distinctive keywords per topic.

Defaults run on the test split (~20K rows) — fast on CPU. Swap to `sample_classical.parquet` for the final report.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from utils import ARTIFACTS_DIR, FIGURES_DIR, set_seed
set_seed()

In [ ]:
df  = pl.read_parquet(ARTIFACTS_DIR / "split_test.parquet").to_pandas()
emb_path = ARTIFACTS_DIR / "emb_test.npy"
if emb_path.exists():
    emb = np.load(emb_path)
    print("Loaded cached embeddings:", emb.shape)
else:
    print("No cached embeddings — recomputing on the fly.")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    emb = model.encode(df["review"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdb_model  = HDBSCAN(min_cluster_size=80, metric="euclidean", prediction_data=True)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdb_model,
    calculate_probabilities=False,
    verbose=True,
)
topics, _ = topic_model.fit_transform(df["review"].tolist(), embeddings=emb)
print("Discovered", len(set(topics)), "topics (incl. noise label -1).")

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(15))
topic_info.to_csv(ARTIFACTS_DIR / "bertopic_topic_info.csv", index=False)

In [ ]:
# Top-N words per top 8 topics
top_topics = topic_info[topic_info["Topic"] != -1].head(8)["Topic"].tolist()
for t in top_topics:
    words = topic_model.get_topic(t)
    print(f"\nTopic {t} (size {topic_info.loc[topic_info.Topic == t, 'Count'].values[0]}):")
    for w, score in words[:8]:
        print(f"  {w:20s} {score:.4f}")

In [ ]:
# BERTopic interactive viz -> static PNG via matplotlib of barchart
try:
    fig = topic_model.visualize_barchart(top_n_topics=8, n_words=8)
    fig.write_html(str(FIGURES_DIR / "07_bertopic_barchart.html"))
    print("Saved interactive barchart to figures/07_bertopic_barchart.html")
except Exception as e:
    print("BERTopic viz skipped:", e)

In [ ]:
# Per-topic mean polarity — finds topics where positive reviews concentrate
df["topic"] = topics
polar = df.groupby("topic")["label"].agg(["mean", "count"]).rename(columns={"mean": "pos_share"})
polar = polar.sort_values("count", ascending=False).head(15)
print(polar)
polar.to_csv(ARTIFACTS_DIR / "bertopic_polarity_per_topic.csv")

In [ ]:
# KeyBERT — sharper keywords than BERTopic's c-TF-IDF in some cases
try:
    from keybert import KeyBERT
    kb = KeyBERT("sentence-transformers/all-MiniLM-L6-v2")
    samples = df.groupby("topic")["review"].apply(lambda s: " ".join(s.head(50))).head(8)
    kb_keywords = {t: kb.extract_keywords(text, top_n=8, keyphrase_ngram_range=(1, 2))
                   for t, text in samples.items()}
    for t, kws in kb_keywords.items():
        print(f"Topic {t}: {[k for k, _ in kws]}")
except ImportError:
    print("KeyBERT not installed; skipping.")

In [ ]:
# Optional: LDA comparison (Gensim) on the same texts, briefly
try:
    from gensim.corpora import Dictionary
    from gensim.models import LdaModel
    docs = [r.split() for r in df["review"].head(5000).tolist()]
    dct = Dictionary(docs); dct.filter_extremes(no_below=10, no_above=0.5)
    corpus = [dct.doc2bow(d) for d in docs]
    lda = LdaModel(corpus, num_topics=8, id2word=dct, passes=2, random_state=42)
    print("\nLDA top words per topic:")
    for i, t in lda.print_topics(num_words=6):
        print(f"  Topic {i}: {t}")
except ImportError:
    print("gensim not installed; skipping LDA comparison.")

In [ ]:
df[["label", "topic", "review"]].to_parquet(ARTIFACTS_DIR / "topics_test.parquet")
print("Saved topics_test.parquet")

## Findings to record

- BERTopic topic count and a 5–8 row table of top-words-per-topic (file: `bertopic_topic_info.csv`).
- Per-topic mean polarity table (file: `bertopic_polarity_per_topic.csv`) — surfaces *which themes are most loaded with positive desire language*.
- LDA top-words for the same N topics — comparison for the Discussion.